Discard any warning messages. Important that BERT is loaded

In [1]:
from transformers import AutoTokenizer, AutoModel

import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Example of a word as a vector

In [2]:
#Give a sample text
text = "Machine learning is fascinating."

#tokenize the text and convert it to input IDs
inputs = tokenizer(text, return_tensors="pt")

#Get the embeddings from the model
with torch.no_grad():
    outputs = model(**inputs)

#Get the embeddings from the last hidden state
embeddings = outputs.last_hidden_state

#Get the embedding for the word "machine" (index 1 in input ids))
machine_embedding = embeddings[0, 1]

print(machine_embedding)

print(machine_embedding.shape)

tensor([ 2.3510e-01,  6.4180e-02, -1.8591e-01, -2.1152e-01,  7.7759e-01,
        -2.3044e-01, -1.8005e-01,  2.5940e-01,  9.3067e-03, -8.0508e-01,
         7.2746e-02, -6.3575e-02, -5.3152e-01, -8.9781e-02, -5.5178e-01,
        -2.3541e-01, -4.3924e-01,  1.2719e-01, -1.1879e-01,  7.0744e-02,
        -3.4261e-01,  8.3972e-02, -4.8617e-01,  4.8519e-01,  1.2489e-01,
         1.2816e-01, -2.0318e-01,  3.0680e-01, -1.1916e-01, -1.3288e-01,
         2.1510e-01, -9.0615e-03, -2.7119e-01, -5.1108e-01, -1.5927e-01,
         2.0769e-01, -6.2177e-01,  3.8358e-01, -2.6007e-01,  3.0104e-01,
         7.4522e-01, -8.1858e-02,  1.0466e-02,  1.8216e-01,  7.6472e-01,
         3.5564e-03, -9.8932e-01, -5.1985e-01,  1.0031e+00, -4.5864e-01,
        -7.5111e-01, -2.2985e-01,  4.8776e-01,  2.6765e-01,  3.2401e-01,
         6.6101e-01,  3.7836e-02,  7.9067e-01,  6.4364e-01, -3.0840e-01,
        -2.3181e-01,  5.2893e-01,  3.9860e-01, -9.0061e-01, -1.1404e-01,
         3.5281e-01, -1.8243e-01, -1.6779e-01, -1.7

Define a function to convert words to embeddings

In [3]:
def get_embedding(word):
    inputs = tokenizer(word, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[0, 1]

    return embedding

In [4]:
print(get_embedding("machine"))

tensor([-2.2897e-01,  7.8094e-03, -9.2770e-03, -2.5354e-01,  9.1216e-01,
         7.9376e-01,  2.2705e-01, -2.9532e-01, -9.5584e-01, -9.9855e-01,
        -2.0191e-01,  7.4684e-01, -7.6690e-02,  6.8413e-01, -2.7934e-01,
         4.6792e-01, -1.5879e-01,  2.3636e-01,  2.4722e-01,  5.5832e-01,
        -5.9263e-01,  3.6152e-01,  2.9228e-01,  9.3172e-02, -4.1154e-02,
         3.4793e-01, -5.5904e-01, -3.2408e-01, -6.6161e-01,  4.1192e-01,
         4.3298e-01, -4.2132e-01,  5.6793e-01,  2.7140e-02, -5.3542e-01,
        -6.0919e-01,  4.2704e-01,  5.2681e-01, -2.7338e-01, -1.0582e-01,
         3.7906e-01, -2.3691e-01,  1.8723e-01, -4.7583e-01,  4.2277e-01,
         4.1130e-01, -1.0440e+00, -1.2717e-01, -2.4032e-01, -4.1772e-02,
        -1.0198e+00,  1.1054e-01,  1.0571e-01,  3.5534e-01, -2.2478e-01,
        -1.4221e-01,  5.9214e-01,  3.8595e-01, -7.6706e-01, -7.2998e-01,
         1.7752e-01,  1.1611e-01,  3.3131e-01,  1.6214e-02,  8.2042e-02,
         3.5110e-01,  6.6074e-01,  2.9693e-01, -1.0

Notice that in this example, "machine" returns a different vector than the previous code cell. This is because of the difference in context. What machine means in this sentence vs. when it is by itself. 

Now we can perform math using words and use cosine similarity to determine what the closest match is

In [5]:
king = get_embedding("king")

woman = get_embedding("woman")

man = get_embedding("man")

result = king + woman - man
    
queen = get_embedding("queen")

similarity = torch.cosine_similarity(result, queen, dim=0)

print(similarity)

tensor(0.7425)


Here, we can intuitevely predict what the word might be without cosine similarity. We have evolved to make these generalizations, however a model is trained on millions (some billions and trillions) of data/tokens. In the previous example, we saw the similarity between the mathematical result and the word which we logically believed would be the result. However, what if that wasn't the case and we wanted to extract the top k words that are similar to the result?

BERT isn't able to satisfy such a linear analogy, however Word2vec from Google can

In [6]:
import sys

print(sys.executable)

print(sys.version)

c:\Users\mckin\OneDrive\Desktop\jupyterbinder\ats\LLM-Tokenizer-From-Scratch\LLMTKN\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [11]:
import gensim.downloader as api

model = api.load('word2vec-google-news-300')

word_to_vec = model

In [12]:
print(word_to_vec['king'])

[ 1.25976562e-01  2.97851562e-02  8.60595703e-03  1.39648438e-01
 -2.56347656e-02 -3.61328125e-02  1.11816406e-01 -1.98242188e-01
  5.12695312e-02  3.63281250e-01 -2.42187500e-01 -3.02734375e-01
 -1.77734375e-01 -2.49023438e-02 -1.67968750e-01 -1.69921875e-01
  3.46679688e-02  5.21850586e-03  4.63867188e-02  1.28906250e-01
  1.36718750e-01  1.12792969e-01  5.95703125e-02  1.36718750e-01
  1.01074219e-01 -1.76757812e-01 -2.51953125e-01  5.98144531e-02
  3.41796875e-01 -3.11279297e-02  1.04492188e-01  6.17675781e-02
  1.24511719e-01  4.00390625e-01 -3.22265625e-01  8.39843750e-02
  3.90625000e-02  5.85937500e-03  7.03125000e-02  1.72851562e-01
  1.38671875e-01 -2.31445312e-01  2.83203125e-01  1.42578125e-01
  3.41796875e-01 -2.39257812e-02 -1.09863281e-01  3.32031250e-02
 -5.46875000e-02  1.53198242e-02 -1.62109375e-01  1.58203125e-01
 -2.59765625e-01  2.01416016e-02 -1.63085938e-01  1.35803223e-03
 -1.44531250e-01 -5.68847656e-02  4.29687500e-02 -2.46582031e-02
  1.85546875e-01  4.47265

King + Woman - Man = ?

In [14]:
print(word_to_vec.most_similar(positive=['woman', 'king'], negative=['man']))

[('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581), ('kings', 0.5236844420433044), ('Queen_Consort', 0.5235945582389832), ('queens', 0.518113374710083), ('sultan', 0.5098593235015869), ('monarchy', 0.5087411403656006)]


In [16]:
print(word_to_vec.similarity('king', 'queen'))
print(word_to_vec.similarity('man', 'woman'))
print(word_to_vec.similarity('king', 'man'))
print(word_to_vec.similarity('queen', 'woman'))

0.6510956
0.76640123
0.22942673
0.31618133


example sentence: quick fox is in the house

suppose in this instance, each word is a token

sort sentence: fox house in is quick the

In [ ]:
input_ids = torch.tensor([2,3,5,1])

vocab_size = 6

output_dimension = 3

torch.manual_seed(123)

embedding_layer = torch.nn.Embedding(num_embeddings=vocab_size, embedding_dim=output_dimension)

print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


Displayed above is a weight matrix of the embedding layer. Size is vocab size x dimensions. This matrix is optimized in training and each row corresponds to the embedding of that specific token